# 場面の時刻を当てる課題: 学習前の 4B の基準値（Colab A100）

用途を「いつ・何が映っているか」に絞ったので、ベンチマーク 800 問すべてを「この問いの場面は動画の何分何秒か」として解かせ、正解区間 ± 許容幅に入る割合を測る。
詳細は `docs/scene_search.md`。

前提: Drive の `MyDrive/movie_llm/` に `frames_f32_s640.zip`（と、あれば 64 フレーム版）がある。
32 フレームで 722 問、約 20 分。終わったら `MyDrive/movie_llm/results/time-4b_f32/predictions.jsonl` を手元の `results/time-4b_f32/` に置く。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "peft>=0.15" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps

## 2. Drive とフレーム

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
import os
!mkdir -p /content/frames
!unzip -q -o {DRIVE}/frames_f32_s640.zip -d /content/frames
os.makedirs('/content/videos', exist_ok=True)
for vid in os.listdir('/content/frames'):
    open(f'/content/videos/{vid}.mp4', 'a').close()
print('フレーム', len(os.listdir('/content/frames')), '本')

## 3. 全 800 問（動画のある 722 問）で時刻を当てる（32 フレーム）

In [ ]:
%cd /content/movie_llm
!python scripts/jvqa_predict.py --name time-4b_f32 --task time --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --max-new-tokens 96 \
    --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results

## 4. 採点（手元でもできる）

In [ ]:
%cd /content/movie_llm
!python scripts/jvqa_eval_time.py --predictions {DRIVE}/results/time-4b_f32/predictions.jsonl --name time-4b_f32 --out-root {DRIVE}/results

終わったら手順 4 の表を手元の私に貼る。`predictions.jsonl` は `results/time-4b_f32/` に置く。

余力があれば、64 フレーム版（`frames_f64_s640.zip` を Drive に置いてから）も `--frames 64 --name time-4b_f64` で測る。temporal の読み取りはフレーム数で伸びるので、時刻の課題も伸びる見込み。